# 04 – EDA: Predictive Maintenance
10.000 synthetische Maschinenzustände mit seltenen Ausfällen. Im Mittelpunkt stehen Datentypen, Verteilungen und Klassenungleichgewicht. **Kein Modelltraining.** Lizenz: CC BY 4.0.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def datenpfad(name):
    kandidaten=[Path("daten")/name,Path("set3-datenanalyse-und-visualisierung/daten")/name,Path("../daten")/name]
    return next(p for p in kandidaten if p.exists())

maschinen=pd.read_csv(datenpfad("ai4i_predictive_maintenance.csv"))
maschinen.head()

## Struktur
UDI und Product ID sind Identifikatoren, Type ist kategorial, fünf Spalten sind Messwerte und Machine failure beschreibt den bekannten Ausfallstatus.

In [ ]:
print("Form:",maschinen.shape)
print("\nDatentypen:\n",maschinen.dtypes)
print("\nFehlende Werte:",int(maschinen.isna().sum().sum()))
display(maschinen.describe(include="all").T)

## Seltene Ausfälle

In [ ]:
anzahl=maschinen["Machine failure"].value_counts().sort_index()
anteil=maschinen["Machine failure"].value_counts(normalize=True).sort_index()*100
display(pd.DataFrame({"Anzahl":anzahl,"Anteil Prozent":anteil.round(2)}))
ax=anzahl.rename(index={0:"kein Ausfall",1:"Ausfall"}).plot.bar(color=["steelblue","firebrick"],figsize=(7,4),title="Klassenverteilung")
ax.set_ylabel("Samples"); ax.set_xlabel(""); plt.show()

## Technische Messwerte

In [ ]:
messwerte=["Air temperature [K]","Process temperature [K]","Rotational speed [rpm]","Torque [Nm]","Tool wear [min]"]
maschinen[messwerte].hist(figsize=(12,8),bins=30,color="darkcyan",edgecolor="white")
plt.suptitle("Verteilungen der Messwerte",y=1.02); plt.tight_layout(); plt.show()

In [ ]:
typen=maschinen.groupby("Type").agg(Samples=("UDI","size"),Ausfaelle=("Machine failure","sum"),Ausfallrate=("Machine failure","mean"))
typen["Ausfallrate Prozent"]=typen["Ausfallrate"]*100
display(typen.round(2))
ax=typen["Ausfallrate Prozent"].plot.bar(color="slateblue",figsize=(7,4),title="Ausfallrate nach Produkttyp")
ax.set_ylabel("Ausfallrate [%]"); plt.show()

## Vergleich und Korrelation

In [ ]:
display(maschinen.groupby("Machine failure")[messwerte].agg(["mean","median"]).round(2))
fig,axes=plt.subplots(1,2,figsize=(12,5))
maschinen.boxplot(column="Torque [Nm]",by="Machine failure",ax=axes[0])
maschinen.boxplot(column="Tool wear [min]",by="Machine failure",ax=axes[1])
plt.suptitle(""); plt.tight_layout(); plt.show()

korrelation=maschinen[messwerte+["Machine failure"]].corr()
display(korrelation.round(2))

Der Datensatz ist synthetisch. Ausfallarten wurden teilweise aus Regeln der Messwerte erzeugt; Zusammenhänge dürfen nicht ungeprüft auf echte Maschinen übertragen werden. IDs sind keine technischen Messwerte.